# 전세가율 분석 노트북

매매 실거래 데이터를 기반으로 **합성 전세 데이터**를 생성하고, 지역별 전세가율을 분석한다.

1. CSV 로드 및 전처리
2. 합성 전세 데이터 생성 (매매가 x 0.5~0.65 비율)
3. 지역별 전세가율 계산 및 시각화
4. 안전/주의/위험 구간 색상 코딩 막대 차트
5. 매매가 vs 전세가 산점도

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline

In [ ]:
# CSV 로드
FILE_PATH = "../data/sample_transactions.csv"

try:
    df = pd.read_csv(FILE_PATH)
    print("데이터 크기:", df.shape)
    display(df.head())
except FileNotFoundError:
    print(f"파일을 찾을 수 없습니다: {FILE_PATH}")
    print("경로를 확인해 주세요.")
    raise

In [ ]:
# 거래금액 숫자 변환
df['거래금액'] = (
    df['거래금액']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(int)
)

print("거래금액 통계:")
df['거래금액'].describe()

In [ ]:
# 합성 전세 데이터 생성
# 실제 전세 데이터가 없으므로, 매매가에 0.5~0.65 비율을 무작위로 적용하여 생성한다.
np.random.seed(42)  # 재현 가능성을 위한 시드 고정

# 각 거래 건에 대해 0.50~0.65 사이 랜덤 비율 생성
df['전세비율_적용'] = np.random.uniform(0.50, 0.65, size=len(df))
df['전세가(합성)'] = (df['거래금액'] * df['전세비율_적용']).astype(int)
df['전세가율'] = (df['전세가(합성)'] / df['거래금액'] * 100).round(1)

print("합성 전세 데이터 미리보기:")
df[['시군구', '법정동', '아파트', '거래금액', '전세가(합성)', '전세가율']].head(10)

In [ ]:
# 지역(시군구)별 평균 전세가율 계산
region_ratio = (
    df.groupby('시군구')['전세가율']
    .mean()
    .sort_values(ascending=False)
    .round(1)
)

print("시군구별 평균 전세가율 (%)")
region_ratio

In [ ]:
# 전세가율 구간별 색상 코딩 막대 차트
# - 안전(녹색): 50% 미만
# - 주의(노란색): 50% ~ 60%
# - 위험(빨간색): 60% 이상

def get_color(ratio):
    """전세가율 구간에 따라 색상을 반환한다."""
    if ratio < 50:
        return '#2ecc71'  # 안전 (녹색)
    elif ratio < 60:
        return '#f39c12'  # 주의 (노란색)
    else:
        return '#e74c3c'  # 위험 (빨간색)

colors = [get_color(v) for v in region_ratio.values]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(region_ratio.index, region_ratio.values, color=colors, edgecolor='white')

# 각 막대 위에 수치 표시
for bar, val in zip(bars, region_ratio.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val}%', ha='center', va='bottom', fontsize=9)

ax.set_title('시군구별 평균 전세가율', fontsize=14, fontweight='bold')
ax.set_ylabel('전세가율 (%)')
ax.set_xlabel('시군구')
ax.axhline(y=50, color='#f39c12', linestyle='--', alpha=0.7, label='주의 기준선 (50%)')
ax.axhline(y=60, color='#e74c3c', linestyle='--', alpha=0.7, label='위험 기준선 (60%)')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 법정동별 전세가율 TOP 15
dong_ratio = (
    df.groupby('법정동')['전세가율']
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .round(1)
)

colors_dong = [get_color(v) for v in dong_ratio.values]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(dong_ratio.index, dong_ratio.values, color=colors_dong, edgecolor='white')

for bar, val in zip(bars, dong_ratio.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val}%', ha='left', va='center', fontsize=9)

ax.set_title('법정동별 평균 전세가율 TOP 15', fontsize=14, fontweight='bold')
ax.set_xlabel('전세가율 (%)')
ax.axvline(x=50, color='#f39c12', linestyle='--', alpha=0.7, label='주의 기준선')
ax.axvline(x=60, color='#e74c3c', linestyle='--', alpha=0.7, label='위험 기준선')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# 매매가 vs 전세가 산점도
fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(
    df['거래금액'],
    df['전세가(합성)'],
    c=df['전세가율'],
    cmap='RdYlGn_r',
    alpha=0.6,
    edgecolors='grey',
    linewidths=0.3,
    s=30
)

# 기준선: 전세가율 50%, 60%, 70%
x_range = np.linspace(df['거래금액'].min(), df['거래금액'].max(), 100)
ax.plot(x_range, x_range * 0.5, '--', color='#2ecc71', alpha=0.8, label='50% (안전)')
ax.plot(x_range, x_range * 0.6, '--', color='#f39c12', alpha=0.8, label='60% (주의)')
ax.plot(x_range, x_range * 0.7, '--', color='#e74c3c', alpha=0.8, label='70% (위험)')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('전세가율 (%)')

ax.set_title('매매가 vs 전세가 산점도', fontsize=14, fontweight='bold')
ax.set_xlabel('매매가 (만원)')
ax.set_ylabel('전세가 (만원, 합성)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 분석 요약

- 합성 전세 데이터는 매매가의 50~65% 범위에서 무작위 생성하였다.
- 실제 분석 시에는 국토교통부 전/월세 실거래 데이터를 활용해야 한다.
- 전세가율이 60%를 넘는 지역은 **갭투자 위험 지역**으로 모니터링이 필요하다.
- 전세가율이 70% 이상이면 **역전세 리스크**가 높아 주의가 필요하다.